# 20 — Publicar para o Streamlit

O Streamlit **nao le o MinIO nem o Nessie diretamente**. Ele consulta o Dremio por Arrow Flight
(porta 32010), e o Dremio le a tabela Iceberg pela fonte `nessie`.

```
refinamento.* (Iceberg no MinIO)  →  Dremio  →  Arrow Flight :32010  →  Streamlit
```

Este notebook confere que a ponte funciona: a tabela existe no catalogo, o Dremio enxerga, e a
consulta volta pelo mesmo caminho que o dashboard usa.

In [ ]:
from lakehouse import sessao, listar

spark = sessao("20-publicar")
listar(spark, "refinamento")

## O Dremio ja enxerga?

O Dremio detecta tabela nova do Nessie sozinho. Se demorar, force o refresh na UI:
fonte `nessie` → engrenagem → **Refresh metadata**.

In [ ]:
import os
from dremio_simple_query.connect import DremioConnection, get_token
from dotenv import load_dotenv

load_dotenv("/workspace/streamlit/vars.env")

token = get_token(
    uri=f"http://{os.getenv('DREMIO_ENDPOINT')}/apiv2/login",
    payload={"userName": os.getenv("DREMIO_USERNAME"),
             "password": os.getenv("DREMIO_PASSWORD")},
)
dremio = DremioConnection(token, f"grpc://{os.getenv('DREMIO_FLIGHT_ENDPOINT')}")
print("conectado ao Dremio via Arrow Flight")

In [ ]:
TABELA = "nessie.refinamento.clientes_por_dominio"

df = dremio.toPandas(f"SELECT * FROM {TABELA}")
print(f"{len(df)} linhas — este e exatamente o DataFrame que o Streamlit recebe\n")
df

## Consultar do Streamlit

Copie o trecho abaixo para o seu `app.py`. O modelo completo, com filtros e cache, esta em
`streamlit_test_jupyter/app.py`.

```python
import os, streamlit as st
from dremio_simple_query.connect import DremioConnection, get_token
from dotenv import load_dotenv

load_dotenv("vars.env")

@st.cache_resource
def conectar():
    token = get_token(uri=f"http://{os.getenv('DREMIO_ENDPOINT')}/apiv2/login",
                      payload={"userName": os.getenv("DREMIO_USERNAME"),
                               "password": os.getenv("DREMIO_PASSWORD")})
    return DremioConnection(token, f"grpc://{os.getenv('DREMIO_FLIGHT_ENDPOINT')}")

@st.cache_data(ttl=300)
def consultar(sql):
    return conectar().toPandas(sql)

df = consultar("SELECT * FROM nessie.refinamento.clientes_por_dominio")
st.dataframe(df)
```

`@st.cache_resource` guarda a conexao (uma so por sessao) e `@st.cache_data` guarda o resultado por
5 minutos — sem isso, cada clique na tela dispara uma consulta nova.

## Rodar o dashboard

```bash
docker compose exec -w /workspace/streamlit spark \
  streamlit run app.py --server.port 8501 --server.address 0.0.0.0
```

Acesse <http://localhost:8502>.

## Se a consulta estiver lenta

Crie uma **reflection** no Dremio: fonte ou dataset → **Reflections** → *Raw*. O Dremio passa a
manter uma copia materializada e responde sem reler o Iceberg. Vale quando a tabela e grande e o
dashboard e consultado com frequencia.